# AllSci Application Crawler & Field Mapper

This notebook performs comprehensive crawling of the entire AllSci application to:
1. **Discover all pages** (list pages, detail pages, tabs)
2. **Build application sitemap** (structure and relationships)
3. **Map all data fields** on every page and tab
4. **Generate comprehensive documentation** of the application structure

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urlparse, urljoin, parse_qs
import json
from datetime import datetime
from collections import defaultdict, deque
import hashlib

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Login credentials
SUPABASE_EMAIL = "rlalani@allsci.com"
SUPABASE_PASSWORD = "!!Casio1994$$"

# Base configuration
BASE_URL = "https://app.allsci.com"
LOGIN_URL = "https://app.allsci.com/?login=true"

# Seed URLs - starting points for crawling
SEED_URLS = [
    "https://app.allsci.com/clinical-trial/",
    "https://app.allsci.com/explore/clinical-trials",
    # Add more seed URLs as needed:
    # "https://app.allsci.com/works/",
    # "https://app.allsci.com/patents/",
    # "https://app.allsci.com/hypotheses/",
]

# Crawling limits
MAX_PAGES_TO_CRAWL = 50  # Limit total pages (increase for full crawl)
MAX_DETAIL_PAGES_PER_LIST = 5  # How many detail pages to visit from each list
MAX_DEPTH = 3  # How many levels deep to crawl

# Wait times
PAGE_LOAD_WAIT = 15  # seconds
ELEMENT_WAIT = 10    # seconds
TAB_SWITCH_WAIT = 3  # seconds

# URL patterns to identify page types
PAGE_TYPE_PATTERNS = {
    'clinical_trial_list': r'/clinical-trial/?$',
    'clinical_trial_detail': r'/clinical-trial/ASC-CT-\d+',
    'explore_atlas': r'/explore/clinical-trials',
    'works_list': r'/works/?$',
    'work_detail': r'/work/ASC-WK-\d+',
    'patent_list': r'/patents/?$',
    'patent_detail': r'/patent/ASC-PT-\d+',
    'hypothesis_list': r'/hypotheses/?$',
    'hypothesis_detail': r'/hypothesis/ASC-HY-\d+',
}

# Tab/section patterns (common tab names)
TAB_PATTERNS = [
    'Overview',
    'Works',
    'Hypotheses',
    'Patents',
    'Related Trials',
    'Timeline',
    'Organizations',
    'People',
    'Funding',
]

## Helper Functions

In [ ]:
def setup_driver(headless=False):
    """Setup Chrome WebDriver."""
    chrome_options = Options()
    if headless:
        chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def login_to_application(driver, login_url, email, password):
    """Login to the application."""
    print(f"Logging in to: {login_url}")
    driver.get(login_url)
    
    try:
        email_input = WebDriverWait(driver, ELEMENT_WAIT).until(
            EC.presence_of_element_located((By.NAME, "email"))
        )
        email_input.send_keys(email)
        
        password_input = driver.find_element(By.NAME, "password")
        password_input.send_keys(password)
        
        sign_in_button = driver.find_element(By.XPATH, "//button[@type='submit' and contains(text(), 'Sign In')]")
        sign_in_button.click()
        
        time.sleep(5)
        print("✓ Login successful!")
        return True
        
    except Exception as e:
        print(f"✗ Login failed: {e}")
        return False


def wait_for_page_load(driver, timeout=PAGE_LOAD_WAIT):
    """Wait for page to finish loading."""
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: d.execute_script('return document.readyState') == 'complete'
        )
        time.sleep(2)
    except TimeoutException:
        pass


def normalize_url(url, base_url=BASE_URL):
    """Normalize URL for comparison."""
    # Remove fragments
    url = url.split('#')[0]
    # Remove trailing slashes for consistency
    url = url.rstrip('/')
    # Make absolute
    if not url.startswith('http'):
        url = urljoin(base_url, url)
    return url


def get_page_type(url):
    """Identify page type based on URL pattern."""
    for page_type, pattern in PAGE_TYPE_PATTERNS.items():
        if re.search(pattern, url):
            return page_type
    return 'unknown'


def get_url_hash(url):
    """Generate a hash for URL deduplication."""
    return hashlib.md5(normalize_url(url).encode()).hexdigest()[:12]

## Link Discovery Functions

In [ ]:
def discover_links(driver, base_url=BASE_URL):
    """Discover all internal links on the current page."""
    links = set()
    
    try:
        # Get all anchor tags
        anchor_elements = driver.find_elements(By.TAG_NAME, "a")
        
        for anchor in anchor_elements:
            try:
                href = anchor.get_attribute('href')
                if href:
                    # Normalize and check if internal
                    normalized = normalize_url(href, base_url)
                    if normalized.startswith(base_url):
                        links.add(normalized)
            except StaleElementReferenceException:
                continue
    except Exception as e:
        print(f"    Warning: Error discovering links: {e}")
    
    return links


def discover_tabs(driver):
    """Discover tabs/sections on the current page."""
    tabs = []
    
    # Common tab selectors
    tab_selectors = [
        "button[role='tab']",
        "a[role='tab']",
        "div[role='tab']",
        ".MuiTab-root",
        "[class*='tab']",
    ]
    
    for selector in tab_selectors:
        try:
            tab_elements = driver.find_elements(By.CSS_SELECTOR, selector)
            
            for tab in tab_elements:
                try:
                    tab_text = tab.text.strip()
                    if tab_text and len(tab_text) < 50:  # Reasonable tab name length
                        # Check if it's a known tab pattern
                        if any(pattern.lower() in tab_text.lower() for pattern in TAB_PATTERNS):
                            tabs.append({
                                'name': tab_text,
                                'element': tab,
                                'selector': selector
                            })
                except:
                    continue
        except:
            continue
    
    # Deduplicate by name
    seen_names = set()
    unique_tabs = []
    for tab in tabs:
        if tab['name'] not in seen_names:
            seen_names.add(tab['name'])
            unique_tabs.append(tab)
    
    return unique_tabs

## Field Extraction Functions

In [ ]:
def get_css_selector(element, driver):
    """Generate CSS selector for an element."""
    try:
        selector = driver.execute_script("""
            function getCssPath(el) {
                if (!(el instanceof Element)) return;
                var path = [];
                while (el.nodeType === Node.ELEMENT_NODE) {
                    var selector = el.nodeName.toLowerCase();
                    if (el.id) {
                        selector += '#' + el.id;
                        path.unshift(selector);
                        break;
                    } else {
                        var sib = el, nth = 1;
                        while (sib = sib.previousElementSibling) {
                            if (sib.nodeName.toLowerCase() == selector)
                                nth++;
                        }
                        if (nth != 1)
                            selector += ":nth-of-type("+nth+")";
                    }
                    path.unshift(selector);
                    el = el.parentNode;
                }
                return path.join(" > ");
            }
            return getCssPath(arguments[0]);
        """, element)
        return selector if selector else "Unknown"
    except:
        return "Unknown"


def extract_metadata_fields(driver):
    """Extract metadata fields."""
    fields = []
    
    try:
        # Look for metadata sections
        metadata_selectors = [
            "div#metadata-content h1",
            "div[class*='metadata'] h1",
            "div[class*='Metadata'] h1",
        ]
        
        for selector in metadata_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                
                for element in elements:
                    text = element.text.strip()
                    if ':' in text:
                        label, value = text.split(':', 1)
                        label = label.strip()
                        value = value.strip()
                        
                        try:
                            value_span = element.find_element(By.TAG_NAME, "span")
                            value = value_span.text.strip()
                        except:
                            pass
                        
                        fields.append({
                            'category': 'metadata',
                            'label': label,
                            'value': value,
                            'selector': get_css_selector(element, driver),
                            'element_type': 'h1',
                            'has_data': bool(value)
                        })
            except:
                continue
    except Exception as e:
        pass
    
    return fields


def extract_all_labeled_fields(driver):
    """Extract all label-value pairs from the page."""
    fields = []
    
    try:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Find common label-value patterns
        # Pattern 1: Label followed by value in separate elements
        for elem in soup.find_all(['dt', 'label', 'span', 'div']):
            text = elem.get_text(strip=True)
            if text and ':' in text and len(text) < 100:
                parts = text.split(':', 1)
                if len(parts) == 2:
                    label, value = parts
                    label = label.strip()
                    value = value.strip()
                    
                    if label and value:
                        fields.append({
                            'category': 'labeled_field',
                            'label': label,
                            'value': value[:200],
                            'selector': f"{elem.name}.{elem.get('class', [''])[0] if elem.get('class') else ''}",
                            'element_type': elem.name,
                            'has_data': True
                        })
    except Exception as e:
        pass
    
    return fields


def extract_button_metrics(driver):
    """Extract metrics from buttons."""
    fields = []
    
    try:
        buttons = driver.find_elements(By.CSS_SELECTOR, "button[aria-label]")
        
        for button in buttons:
            try:
                aria_label = button.get_attribute('aria-label')
                text = button.text.strip()
                
                if text and (text.isdigit() or re.search(r'\d+', text)):
                    fields.append({
                        'category': 'button_metric',
                        'label': aria_label,
                        'value': text,
                        'selector': get_css_selector(button, driver),
                        'element_type': 'button',
                        'has_data': True
                    })
            except:
                continue
    except Exception as e:
        pass
    
    return fields


def extract_table_data(driver):
    """Extract data from tables."""
    fields = []
    
    try:
        tables = driver.find_elements(By.TAG_NAME, "table")
        
        for idx, table in enumerate(tables):
            try:
                headers = []
                header_cells = table.find_elements(By.TAG_NAME, "th")
                headers = [h.text.strip() for h in header_cells if h.text.strip()]
                
                if headers:
                    for header in headers:
                        fields.append({
                            'category': 'table_header',
                            'label': f"Table {idx + 1} - {header}",
                            'value': '',
                            'selector': get_css_selector(table, driver),
                            'element_type': 'table',
                            'has_data': False
                        })
            except:
                continue
    except Exception as e:
        pass
    
    return fields


def extract_heading_structure(driver):
    """Extract page heading structure."""
    fields = []
    
    try:
        for level in ['h1', 'h2', 'h3']:
            headings = driver.find_elements(By.TAG_NAME, level)
            for heading in headings:
                text = heading.text.strip()
                if text and len(text) < 200:
                    fields.append({
                        'category': f'heading_{level}',
                        'label': text,
                        'value': '',
                        'selector': get_css_selector(heading, driver),
                        'element_type': level,
                        'has_data': False
                    })
    except Exception as e:
        pass
    
    return fields


def extract_all_fields(driver, page_url, tab_name=None):
    """Extract all fields from the current page/tab."""
    all_fields = []
    
    # Extract different types
    all_fields.extend(extract_metadata_fields(driver))
    all_fields.extend(extract_all_labeled_fields(driver))
    all_fields.extend(extract_button_metrics(driver))
    all_fields.extend(extract_table_data(driver))
    all_fields.extend(extract_heading_structure(driver))
    
    # Add context to each field
    for field in all_fields:
        field['page_url'] = page_url
        field['page_title'] = driver.title
        field['tab_name'] = tab_name
        field['page_type'] = get_page_type(page_url)
    
    return all_fields

## Page Crawling Functions

In [ ]:
def crawl_page_with_tabs(driver, url, depth=0):
    """Crawl a single page including all its tabs."""
    page_data = {
        'url': url,
        'page_type': get_page_type(url),
        'title': '',
        'depth': depth,
        'tabs': [],
        'fields': [],
        'links': set(),
        'crawl_timestamp': datetime.now().isoformat()
    }
    
    try:
        print(f"  {'  ' * depth}Crawling: {url}")
        driver.get(url)
        wait_for_page_load(driver)
        
        page_data['title'] = driver.title
        
        # Extract fields from main page
        print(f"  {'  ' * depth}  - Extracting fields from main page")
        main_fields = extract_all_fields(driver, url)
        page_data['fields'].extend(main_fields)
        
        # Discover tabs
        tabs = discover_tabs(driver)
        if tabs:
            print(f"  {'  ' * depth}  - Found {len(tabs)} tabs: {', '.join([t['name'] for t in tabs])}")
            
            for tab in tabs:
                try:
                    print(f"  {'  ' * depth}    → Clicking tab: {tab['name']}")
                    tab['element'].click()
                    time.sleep(TAB_SWITCH_WAIT)
                    
                    # Extract fields from this tab
                    tab_fields = extract_all_fields(driver, url, tab_name=tab['name'])
                    page_data['fields'].extend(tab_fields)
                    page_data['tabs'].append(tab['name'])
                    
                    print(f"  {'  ' * depth}      Extracted {len(tab_fields)} fields")
                except Exception as e:
                    print(f"  {'  ' * depth}      Warning: Could not process tab {tab['name']}: {e}")
                    continue
        
        # Discover links
        links = discover_links(driver)
        page_data['links'] = links
        print(f"  {'  ' * depth}  - Found {len(links)} links")
        
        print(f"  {'  ' * depth}✓ Total fields extracted: {len(page_data['fields'])}")
        
    except Exception as e:
        print(f"  {'  ' * depth}✗ Error crawling {url}: {e}")
    
    return page_data


def crawl_application(driver, seed_urls, max_pages=MAX_PAGES_TO_CRAWL, max_depth=MAX_DEPTH):
    """Crawl the entire application starting from seed URLs."""
    
    visited = set()
    to_visit = deque([(url, 0) for url in seed_urls])  # (url, depth)
    
    all_pages = []
    all_fields = []
    sitemap = defaultdict(list)  # parent_url -> [child_urls]
    
    page_count = 0
    
    while to_visit and page_count < max_pages:
        url, depth = to_visit.popleft()
        
        # Skip if already visited or too deep
        if url in visited or depth > max_depth:
            continue
        
        visited.add(url)
        page_count += 1
        
        print(f"\n[{page_count}/{max_pages}] Depth {depth}")
        
        # Crawl the page
        page_data = crawl_page_with_tabs(driver, url, depth)
        all_pages.append(page_data)
        all_fields.extend(page_data['fields'])
        
        # Add discovered links to queue
        page_type = page_data['page_type']
        
        # Determine how many links to follow based on page type
        if 'list' in page_type:
            # For list pages, follow limited number of detail pages
            detail_links = [link for link in page_data['links'] 
                          if 'detail' in get_page_type(link)]
            links_to_follow = detail_links[:MAX_DETAIL_PAGES_PER_LIST]
        else:
            # For other pages, follow all internal links
            links_to_follow = page_data['links']
        
        for link in links_to_follow:
            if link not in visited:
                to_visit.append((link, depth + 1))
                sitemap[url].append(link)
        
        # Show progress
        print(f"  Queue size: {len(to_visit)} | Visited: {len(visited)}")
    
    print(f"\n{'='*60}")
    print(f"Crawl Complete!")
    print(f"  Pages crawled: {page_count}")
    print(f"  Total fields extracted: {len(all_fields)}")
    print(f"{'='*60}")
    
    return {
        'pages': all_pages,
        'fields': all_fields,
        'sitemap': dict(sitemap),
        'visited_urls': list(visited)
    }

## Run the Crawler

In [ ]:
# Setup and login
print("Setting up crawler...\n")
driver = setup_driver(headless=False)

# Login
login_success = login_to_application(driver, LOGIN_URL, SUPABASE_EMAIL, SUPABASE_PASSWORD)

if not login_success:
    print("Login failed. Please check credentials.")
    driver.quit()
else:
    print("\nStarting application crawl...\n")
    print(f"{'='*60}")
    print(f"Configuration:")
    print(f"  Max pages: {MAX_PAGES_TO_CRAWL}")
    print(f"  Max depth: {MAX_DEPTH}")
    print(f"  Detail pages per list: {MAX_DETAIL_PAGES_PER_LIST}")
    print(f"  Seed URLs: {len(SEED_URLS)}")
    for url in SEED_URLS:
        print(f"    - {url}")
    print(f"{'='*60}\n")
    
    # Crawl
    crawl_results = crawl_application(driver, SEED_URLS)
    
    # Cleanup
    driver.quit()

## Generate Reports

In [ ]:
# Create DataFrames
df_fields = pd.DataFrame(crawl_results['fields'])
df_pages = pd.DataFrame([{
    'url': p['url'],
    'page_type': p['page_type'],
    'title': p['title'],
    'depth': p['depth'],
    'num_tabs': len(p['tabs']),
    'tabs': ', '.join(p['tabs']),
    'num_fields': len(p['fields']),
    'num_links': len(p['links']),
} for p in crawl_results['pages']])

print("\n" + "="*60)
print("CRAWL SUMMARY")
print("="*60)
print(f"\nTotal Pages: {len(df_pages)}")
print(f"Total Fields: {len(df_fields)}")
print(f"\nPages by Type:")
print(df_pages['page_type'].value_counts())
print(f"\nFields by Category:")
print(df_fields['category'].value_counts())
print(f"\nTop 10 Pages by Field Count:")
print(df_pages.nlargest(10, 'num_fields')[['url', 'page_type', 'num_fields', 'num_tabs']])

## Application Structure Map

In [ ]:
# Generate application structure
print("\n" + "="*60)
print("APPLICATION STRUCTURE MAP")
print("="*60)

for page in crawl_results['pages'][:20]:  # Show first 20
    indent = "  " * page['depth']
    print(f"\n{indent}📄 {page['url'].replace(BASE_URL, '')}")
    print(f"{indent}   Type: {page['page_type']} | Fields: {len(page['fields'])} | Links: {len(page['links'])}")
    if page['tabs']:
        print(f"{indent}   Tabs: {', '.join(page['tabs'])}")

## Field Coverage Analysis

In [ ]:
# Analyze field coverage
print("\n" + "="*60)
print("FIELD COVERAGE BY PAGE TYPE")
print("="*60)

coverage = df_fields.groupby(['page_type', 'category']).agg({
    'label': 'count',
    'has_data': 'sum'
}).rename(columns={'label': 'total_fields', 'has_data': 'fields_with_data'})

print(coverage)

# Show unique field labels by page type
print("\n" + "="*60)
print("UNIQUE FIELD LABELS BY PAGE TYPE")
print("="*60)

for page_type in df_fields['page_type'].unique():
    print(f"\n{page_type.upper()}:")
    type_fields = df_fields[df_fields['page_type'] == page_type]
    unique_labels = type_fields['label'].unique()[:15]
    for label in unique_labels:
        print(f"  - {label}")
    if len(type_fields['label'].unique()) > 15:
        print(f"  ... and {len(type_fields['label'].unique()) - 15} more")

## Tab Analysis

In [ ]:
# Analyze tabs
print("\n" + "="*60)
print("TAB ANALYSIS")
print("="*60)

tab_fields = df_fields[df_fields['tab_name'].notna()]
if len(tab_fields) > 0:
    print(f"\nTotal fields from tabs: {len(tab_fields)}")
    print(f"\nFields by Tab:")
    print(tab_fields.groupby('tab_name')['label'].count().sort_values(ascending=False))
    
    print(f"\nPages with Tabs:")
    pages_with_tabs = df_pages[df_pages['num_tabs'] > 0]
    print(pages_with_tabs[['url', 'page_type', 'tabs', 'num_tabs']].to_string())
else:
    print("\nNo tabs were detected on any pages.")

## Export Results

In [ ]:
# Export all data
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Field mapping CSV
field_file = f'field_mapping_full_{timestamp}.csv'
df_fields.to_csv(field_file, index=False)
print(f"\n✓ Field mapping exported to: {field_file}")

# 2. Page structure CSV
page_file = f'page_structure_{timestamp}.csv'
df_pages.to_csv(page_file, index=False)
print(f"✓ Page structure exported to: {page_file}")

# 3. Complete crawl data JSON
json_file = f'crawl_results_{timestamp}.json'
with open(json_file, 'w') as f:
    json.dump({
        'pages': [{
            **p,
            'links': list(p['links'])  # Convert sets to lists for JSON
        } for p in crawl_results['pages']],
        'sitemap': crawl_results['sitemap'],
        'visited_urls': crawl_results['visited_urls'],
        'summary': {
            'total_pages': len(crawl_results['pages']),
            'total_fields': len(crawl_results['fields']),
            'crawl_timestamp': timestamp
        }
    }, f, indent=2)
print(f"✓ Complete crawl data exported to: {json_file}")

# 4. Sitemap visualization
sitemap_file = f'sitemap_{timestamp}.txt'
with open(sitemap_file, 'w') as f:
    f.write("APPLICATION SITEMAP\n")
    f.write("="*80 + "\n\n")
    
    for page in crawl_results['pages']:
        indent = "  " * page['depth']
        f.write(f"{indent}{page['url']}\n")
        f.write(f"{indent}  Type: {page['page_type']}\n")
        f.write(f"{indent}  Fields: {len(page['fields'])}\n")
        if page['tabs']:
            f.write(f"{indent}  Tabs: {', '.join(page['tabs'])}\n")
        f.write("\n")

print(f"✓ Sitemap exported to: {sitemap_file}")

print(f"\n{'='*60}")
print("All exports complete!")
print(f"{'='*60}")